# GeoLife CP2 — Home / Office / POI Baseline

**Mục tiêu:** xây một baseline Home / Office có thể giải thích được, bắt đầu từ frozen CP1 stay events chứ không từ raw GPS.

### Câu hỏi lớn

1. user có đủ repeated history để suy semantic location không;
2. một “recurring location” nên được gom như thế nào để spatial meaning rõ ràng;
3. UTC timestamps phải chuyển sang behavioral local time theo policy nào;
4. night/daytime evidence nên tính trên whole stay hay exact interval overlap;
5. khi nào nên **abstain** thay vì ép HOME/OFFICE;
6. production implementation có reproduce đúng notebook decision trên full release không.

### Nguyên tắc

```text
CP1 cleaning/stays
      ↓
user-level history
      ↓
geography + timezone scope
      ↓
compact recurring locations
      ↓
behavioral-time evidence
      ↓
abstention + evidence strength
```

> GeoLife không có direct HOME/OFFICE ground truth. Vì vậy notebook đánh giá **coverage, stability, support và plausibility**, không báo accuracy.

> HOME/OFFICE là sensitive derived locations. Không commit precise user-level inferred coordinates hoặc private caches vào repo.

Design contract: `docs/design/03_home_office_baseline_contract.md`.

In [ ]:
from pathlib import Path
from time import perf_counter
from zoneinfo import ZoneInfo
from IPython.display import display
import os
import pickle
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")
CACHE_DIR = VOLUME_ROOT / "cache" / "cp2_home_office"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_root():
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env_root)] if env_root else []) + [
        VOLUME_ROOT / "extracted" / "Geolife Trajectories 1.3" / "Data",
        VOLUME_ROOT / "Data",
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    for candidate in sorted(VOLUME_ROOT.glob("**/Data")):
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    raise FileNotFoundError("GeoLife Data folder not found")

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

DATA_ROOT = resolve_data_root()
ensure_repo()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from notebooks.eda_core import read_plt
from geolife.geo.distance import haversine_m
from geolife.staypoints import clean_trajectory, detect_staypoints
from geolife.model import HomeOfficeConfig, infer_home_office

BASELINE = {
    "same_second_radius_m": 10.0,
    "max_gap_s": 300.0,
    "hard_speed_guard_kmh": 1200.0,
    "distance_threshold_m": 200.0,
    "min_dwell_s": 1200.0,
}

files = sorted(DATA_ROOT.glob("*/Trajectory/*.plt"))
print("Repo branch:", REPO_BRANCH)
print("Data root:", DATA_ROOT)
print("Trajectory files:", f"{len(files):,}")
print("Cache dir:", CACHE_DIR)
print("Frozen CP1 baseline:", BASELINE)

## 1. Materialize frozen CP1 stays ở cấp user

CP1 full-release audit đã xác nhận **5,821 stays** trên 18,670 files, nhưng cache cũ chủ yếu là per-file summary. CP2 cần actual stay rows để gom history theo user.

### Vì sao phải materialize lại actual stays?

Home/Office cần các field mà summary per-file không đủ:

- `user_id`;
- arrival / departure UTC;
- duration;
- stay representative coordinate;
- source-file lineage.

### Reproducibility gate

Section này phải kết thúc bằng:

```text
len(stays) == 5,821
```

Nếu không reconcile đúng CP1 total thì phải dừng — semantic stage không được âm thầm chạy trên một upstream dataset khác.

### Cache design

Run đầu có thể chậm vì phải đọc 18,670 `.plt` files. Notebook checkpoint mỗi 500 files và lưu final private cache bằng pandas pickle để không phụ thuộc `pyarrow`.

Cache chỉ là execution artifact trên mounted volume, không phải dataset để commit.

In [ ]:
STAYS_CACHE = CACHE_DIR / "stays_baseline_v1.pkl"
PARTIAL_CACHE = CACHE_DIR / "stays_baseline_v1.partial.pkl"
EXPECTED_CP1_STAYS = 5821

def user_id_from_path(path):
    return path.parent.parent.name

def process_file(path):
    raw = read_plt(path)[["timestamp", "latitude", "longitude"]]
    cleaned = clean_trajectory(
        raw,
        same_second_radius_m=BASELINE["same_second_radius_m"],
        max_gap_s=BASELINE["max_gap_s"],
        hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
    )
    stays = detect_staypoints(
        cleaned,
        distance_threshold_m=BASELINE["distance_threshold_m"],
        min_dwell_s=BASELINE["min_dwell_s"],
    )
    if stays.empty:
        return []
    user_id = user_id_from_path(path)
    out = []
    for row in stays.itertuples(index=False):
        out.append({
            "user_id": user_id,
            "source_file": str(path),
            "sequence_id": int(row.sequence_id),
            "arrival_time_utc": row.arrival_time,
            "departure_time_utc": row.departure_time,
            "duration_s": float(row.duration_s),
            "latitude": float(row.latitude),
            "longitude": float(row.longitude),
            "n_points": int(row.n_points),
        })
    return out

if STAYS_CACHE.exists():
    stays = pd.read_pickle(STAYS_CACHE)
    print("Loaded:", STAYS_CACHE)
else:
    if PARTIAL_CACHE.exists():
        with PARTIAL_CACHE.open("rb") as f:
            partial = pickle.load(f)
        processed = set(partial["processed_files"])
        rows = list(partial["rows"])
        print("Resuming partial:", f"{len(processed):,}/{len(files):,} files")
    else:
        processed = set()
        rows = []

    t0 = perf_counter()
    completed_this_run = 0

    for path in files:
        key = str(path)
        if key in processed:
            continue

        rows.extend(process_file(path))
        processed.add(key)
        completed_this_run += 1

        if completed_this_run % 500 == 0:
            elapsed_min = (perf_counter() - t0) / 60
            overall_done = len(processed)
            rate = completed_this_run / max(elapsed_min, 1e-9)
            remaining = len(files) - overall_done
            eta_min = remaining / max(rate, 1e-9)
            print(
                f"{overall_done:,}/{len(files):,} files | "
                f"{len(rows):,} stays | "
                f"elapsed {elapsed_min:.1f} min | ETA ~{eta_min:.1f} min"
            )
            with PARTIAL_CACHE.open("wb") as f:
                pickle.dump(
                    {"processed_files": sorted(processed), "rows": rows},
                    f,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )

    stays = pd.DataFrame(rows)
    stays["arrival_time_utc"] = pd.to_datetime(stays["arrival_time_utc"], utc=True)
    stays["departure_time_utc"] = pd.to_datetime(stays["departure_time_utc"], utc=True)
    stays = stays.sort_values(
        ["user_id", "arrival_time_utc", "source_file"], kind="stable"
    ).reset_index(drop=True)
    stays.to_pickle(STAYS_CACHE)
    if PARTIAL_CACHE.exists():
        PARTIAL_CACHE.unlink()
    print("Saved:", STAYS_CACHE)

print("Materialized stays:", f"{len(stays):,}")
print("Users with stays:", stays["user_id"].nunique())
assert len(stays) == EXPECTED_CP1_STAYS, (
    f"Expected {EXPECTED_CP1_STAYS} CP1 stays, got {len(stays)}"
)
display(stays.head())

### Kết luận phần 1

Full run đã reproduce chính xác:

- **5,821 stays**;
- **136 users** có ít nhất một stay.

Điều này đóng contract giữa CP1 và CP2: mọi semantic analysis về sau bắt đầu từ đúng production stay behavior đã validate.

**Không được suy ra:** 136 users có stays không có nghĩa 136 users đủ evidence để infer HOME/OFFICE. History sufficiency là gate riêng ở phần tiếp theo.

## 2. User-level history sufficiency

### Câu hỏi

Một user có 1–2 stays có nên bị ép ra HOME/OFFICE không?

Không. Semantic location là **repeated behavior**, nên insufficient history phải là một trạng thái hợp lệ.

Section này đo:

- stays/user;
- số distinct UTC dates có stay;
- observation span;
- total dwell;
- median stay duration.

> Các date ở đây vẫn là **UTC dates**. Chưa được dùng chúng làm night/weekday behavioral semantics trước timezone gate.

In [ ]:
user_history = (
    stays.assign(
        arrival_utc_date=stays["arrival_time_utc"].dt.date,
    )
    .groupby("user_id")
    .agg(
        stays=("user_id", "size"),
        active_utc_dates=("arrival_utc_date", "nunique"),
        first_stay_utc=("arrival_time_utc", "min"),
        last_stay_utc=("departure_time_utc", "max"),
        total_dwell_h=("duration_s", lambda s: s.sum() / 3600.0),
        median_stay_min=("duration_s", lambda s: s.median() / 60.0),
    )
)

user_history["observation_span_days"] = (
    user_history["last_stay_utc"] - user_history["first_stay_utc"]
).dt.total_seconds() / 86400.0

display(
    user_history[
        ["stays", "active_utc_dates", "observation_span_days", "total_dwell_h", "median_stay_min"]
    ].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99])
)

print("Users with >=1 stay:", len(user_history))
for n in [2, 3, 5, 10]:
    print(f"Users with >= {n} stays:", int((user_history["stays"] >= n).sum()))
for n in [2, 3, 5, 10]:
    print(
        f"Users with stays on >= {n} distinct UTC dates:",
        int((user_history["active_utc_dates"] >= n).sum()),
    )

### Kết luận phần 2

Measured support:

| history condition | users |
|---|---:|
| >=1 stay | **136** |
| >=2 stays | **120** |
| >=5 stays | **99** |
| >=10 stays | **81** |
| stays trên >=2 UTC dates | **114** |
| >=5 UTC dates | **83** |
| >=10 UTC dates | **62** |

Release có 182 users, tức **46 users không có detected stay** dưới frozen CP1 baseline. Ngay trong 136 users còn lại, repeated-history support cũng không đồng đều.

**Decision:** CP2 phải có abstention; coverage của pipeline không được đánh đồng với semantic certainty.

## 3. Historical prototype — DBSCAN recurring locations

Home/Office là thuộc tính của **recurring location theo user**, không phải của một individual stay.

Prototype đầu tiên thử Haversine DBSCAN `eps=200 m`, `min_samples=1` để xem repeated structure.

### Vì sao DBSCAN được thử?

- không cần biết trước số locations/user;
- Haversine distance phù hợp lat/lon;
- 200 m là spatial scale đã quen thuộc từ CP1.

### Nhưng `eps=200 m` không có nghĩa cluster diameter <=200 m

DBSCAN dùng density connectivity:

```text
A --180m-- B --180m-- C --180m-- D
```

Mỗi local edge hợp lệ nhưng A↔D có thể xa hơn nhiều 200 m. Đây là **chaining**.

Vì vậy section này là historical audit/prototype. Production CP2 **không dùng DBSCAN**; complete-linkage ở phần 4.3 thay thế nó bằng một explicit diameter contract.

In [ ]:
EARTH_RADIUS_M = 6_371_008.8
LOCATION_EPS_M = 200.0

def cluster_user_stays(group, eps_m=LOCATION_EPS_M):
    g = group.sort_values("arrival_time_utc", kind="stable").copy()
    coords_rad = np.radians(g[["latitude", "longitude"]].to_numpy(dtype=float))
    labels = DBSCAN(
        eps=eps_m / EARTH_RADIUS_M,
        min_samples=1,
        metric="haversine",
        algorithm="ball_tree",
    ).fit_predict(coords_rad)
    g["location_id"] = labels.astype(int)
    return g

cluster_parts = []
for user_id, group in stays.groupby("user_id", sort=True):
    clustered_user = cluster_user_stays(group)
    cluster_parts.append(clustered_user)

clustered = pd.concat(cluster_parts, ignore_index=True) if cluster_parts else stays.copy()

location_rows = []
for (user_id, location_id), g in clustered.groupby(["user_id", "location_id"], sort=True):
    lat = float(g["latitude"].median())
    lon = float(g["longitude"].median())
    radii = np.asarray(
        haversine_m(
            g["latitude"].to_numpy(dtype=float),
            g["longitude"].to_numpy(dtype=float),
            lat,
            lon,
        ),
        dtype=float,
    )
    location_rows.append({
        "user_id": user_id,
        "location_id": int(location_id),
        "latitude": lat,
        "longitude": lon,
        "stay_count": len(g),
        "active_utc_dates": g["arrival_time_utc"].dt.date.nunique(),
        "total_dwell_h": g["duration_s"].sum() / 3600.0,
        "max_radius_m": float(np.max(radii)) if len(radii) else 0.0,
    })

locations = pd.DataFrame(location_rows)

print("Users:", locations["user_id"].nunique())
print("Candidate locations:", len(locations))
print("Recurring locations (>=2 stays):", int((locations["stay_count"] >= 2).sum()))
print("Users with >=1 recurring location:", locations.loc[
    locations["stay_count"] >= 2, "user_id"
].nunique())

display(
    locations[
        ["stay_count", "active_utc_dates", "total_dwell_h", "max_radius_m"]
    ].describe(percentiles=[.5,.75,.9,.95,.99])
)

display(
    locations.sort_values("max_radius_m", ascending=False).head(20)
)

### Kết luận DBSCAN prototype

Measured:

- **1,885 candidate locations**;
- **635 recurring locations** có >=2 stays;
- **104 users** có ít nhất một recurring location;
- max distance từ median representative tới member ≈ **526.7 m** dù `eps=200 m`.

Đây không phải bug của DBSCAN; đó là đúng semantics của density connectivity.

**Decision:** không freeze DBSCAN cho semantic location. Ta cần representation mà threshold có nghĩa trực tiếp về total compactness.

### Tóm tắt evidence trước timezone gate

Đến đây ta biết:

```text
5,821 stays
136 users có stay
104 users có recurring DBSCAN prototype location
DBSCAN 200m bị chaining tới ~526.7m radius-from-median
```

Spatial summary của stays cho thấy dataset **tập trung mạnh quanh Beijing nhưng không chỉ có Beijing**. Longitude của stays trải rất rộng, nên blanket `UTC + 8h` cho toàn release là không defensible.

Vì vậy hai gate tiếp theo là:

1. explicit Beijing-focused cohort;
2. compact recurring-location representation.

## 4. Timezone / geography audit

GeoLife PLT timestamps là UTC/GMT, nhưng HOME/OFFICE là behavioral-time concepts.

### Failure mode nếu cộng thẳng +8h

Một user có thể chủ yếu ở Beijing nhưng có travel stays ở Tokyo, Seattle hoặc nơi khác. Nếu gán toàn bộ user sang `Asia/Shanghai`, travel observations sẽ có local hour sai.

Do đó timezone policy phải có **hai tầng**:

```text
user-level eligibility  → user có đủ Beijing-focused không?
observation-level scope → stay này có nằm trong Beijing region không?
```

Chỉ in-region stays của eligible users mới được convert sang `Asia/Shanghai` cho v1.

In [ ]:
spatial_summary = stays[["latitude", "longitude"]].describe(
    percentiles=[.01,.05,.25,.5,.75,.95,.99]
)
display(spatial_summary)

user_centers = (
    stays.groupby("user_id")[["latitude", "longitude"]]
    .median()
    .rename(columns={"latitude":"median_latitude","longitude":"median_longitude"})
)
display(user_centers.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

sample_n = min(5000, len(stays))
plot_sample = stays.sample(sample_n, random_state=42) if sample_n else stays
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(plot_sample["longitude"], plot_sample["latitude"], s=8, alpha=0.35)
ax.set(
    title="Stay-point spatial coverage (sample; UTC semantics not yet converted)",
    xlabel="longitude",
    ylabel="latitude",
)
plt.show()

print("TIMEZONE POLICY STATUS: OPEN")
print("Do not run Home/Office time-of-day scoring until this gate is reviewed.")

### 4.1 Beijing-focused geography sensitivity

Reference point dùng để đo distance:

```text
39.9042° N, 116.4074° E
```

Đây chỉ là **distance anchor**, không phải administrative boundary.

Audit radii: **50 / 100 / 200 km**. Với mỗi user, tính hai tỷ lệ:

```text
stay_share_inside  = số stays trong radius / total stays
dwell_share_inside = dwell trong radius / total dwell
```

Ta cần cả hai vì nhiều short stays và một long travel stay có thể cho two views rất khác.

### Measured sensitivity ở threshold 80/80%

| radius | eligible users |
|---:|---:|
| 50 km | 90 |
| **100 km** | **97** |
| 200 km | 103 |

`100 km + 80% stay share + 80% dwell share` nằm giữa sensitivity, nên được freeze như **CP2 v1 engineering cohort**, không phải accuracy-optimal geography boundary.

In [ ]:
BEIJING_CENTER = (39.9042, 116.4074)
BEIJING_RADII_KM = [50.0, 100.0, 200.0]
CANDIDATE_RADIUS_KM = 100.0
CANDIDATE_MIN_STAY_SHARE = 0.80
CANDIDATE_MIN_DWELL_SHARE = 0.80

stays_geo = stays.copy()
stays_geo["distance_to_beijing_km"] = (
    np.asarray(
        haversine_m(
            stays_geo["latitude"].to_numpy(dtype=float),
            stays_geo["longitude"].to_numpy(dtype=float),
            BEIJING_CENTER[0],
            BEIJING_CENTER[1],
        ),
        dtype=float,
    )
    / 1000.0
)

print("Stay distance to Beijing reference point (km):")
display(
    stays_geo["distance_to_beijing_km"].describe(
        percentiles=[.5, .75, .9, .95, .99]
    )
)

radius_rows = []
user_geo_frames = {}

for radius_km in BEIJING_RADII_KM:
    inside = stays_geo["distance_to_beijing_km"] <= radius_km
    tmp = stays_geo.assign(
        inside_radius=inside,
        inside_dwell_s=np.where(inside, stays_geo["duration_s"], 0.0),
    )

    by_user = (
        tmp.groupby("user_id")
        .agg(
            total_stays=("user_id", "size"),
            inside_stays=("inside_radius", "sum"),
            total_dwell_s=("duration_s", "sum"),
            inside_dwell_s=("inside_dwell_s", "sum"),
        )
    )
    by_user["stay_share_inside"] = by_user["inside_stays"] / by_user["total_stays"]
    by_user["dwell_share_inside"] = (
        by_user["inside_dwell_s"] / by_user["total_dwell_s"]
    )
    user_geo_frames[radius_km] = by_user

    for min_share in [0.50, 0.80, 0.90, 0.95]:
        eligible = (
            (by_user["stay_share_inside"] >= min_share)
            & (by_user["dwell_share_inside"] >= min_share)
        )
        radius_rows.append({
            "radius_km": radius_km,
            "min_both_shares": min_share,
            "users": int(eligible.sum()),
            "stays_from_eligible_users": int(
                by_user.loc[eligible, "total_stays"].sum()
            ),
        })

radius_sensitivity = pd.DataFrame(radius_rows)
display(radius_sensitivity)

candidate_geo = user_geo_frames[CANDIDATE_RADIUS_KM].copy()
candidate_geo["beijing_candidate"] = (
    (candidate_geo["stay_share_inside"] >= CANDIDATE_MIN_STAY_SHARE)
    & (candidate_geo["dwell_share_inside"] >= CANDIDATE_MIN_DWELL_SHARE)
)

beijing_user_ids = candidate_geo.index[candidate_geo["beijing_candidate"]]
candidate_user_mask = stays_geo["user_id"].isin(beijing_user_ids)
inside_candidate_radius = (
    stays_geo["distance_to_beijing_km"] <= CANDIDATE_RADIUS_KM
)

# Only in-region stays receive Asia/Shanghai semantic-time treatment.
# Travel/out-of-region stays from otherwise Beijing-focused users remain excluded.
stays_beijing = stays_geo[candidate_user_mask & inside_candidate_radius].copy()
excluded_travel_stays = stays_geo[candidate_user_mask & ~inside_candidate_radius].copy()

print("Candidate Beijing users:", len(beijing_user_ids))
print("In-region candidate stays:", len(stays_beijing))
print("Excluded travel/out-of-region stays from candidate users:", len(excluded_travel_stays))
print(
    "Share of all materialized stays used for Beijing semantic audit:",
    f"{len(stays_beijing) / len(stays_geo):.2%}",
)

display(
    candidate_geo[
        [
            "total_stays",
            "inside_stays",
            "stay_share_inside",
            "dwell_share_inside",
            "beijing_candidate",
        ]
    ]
    .sort_values(
        ["beijing_candidate", "stay_share_inside", "dwell_share_inside"],
        ascending=[False, False, False],
    )
    .head(30)
)

### Kết luận geography gate

Frozen v1 policy cho:

- **97 eligible users**;
- **4,197 in-region stays** dùng cho semantic-time processing;
- **245 travel/out-of-region stays** bị exclude dù user vẫn đủ điều kiện;
- retained semantic stays = **72.10%** của toàn bộ 5,821 stays.

Điểm 245 travel stays rất quan trọng: timezone eligibility được quyết định ở user-level nhưng timezone application vẫn scoped ở **observation-level**.

Users ngoài cohort **abstain** trong v1; notebook không đoán timezone từ longitude.

### 4.2 Local-time conversion cho frozen Beijing cohort

Chỉ 4,197 in-region semantic stays được convert bằng timezone-aware operation:

```text
UTC timestamp
   ↓ tz_convert
Asia/Shanghai local timestamp
```

Không dùng manual `+8h`, vì timezone object giữ semantics rõ ràng và tránh biến conversion thành arithmetic ad-hoc.

`arrival_local_date`, `arrival_local_hour`, `arrival_local_weekday` từ đây mới được phép dùng cho behavioral features.

In [ ]:
BEIJING_TZ = ZoneInfo("Asia/Shanghai")

stays_beijing["arrival_time_local"] = (
    stays_beijing["arrival_time_utc"].dt.tz_convert(BEIJING_TZ)
)
stays_beijing["departure_time_local"] = (
    stays_beijing["departure_time_utc"].dt.tz_convert(BEIJING_TZ)
)
stays_beijing["arrival_local_date"] = stays_beijing["arrival_time_local"].dt.date
stays_beijing["arrival_local_hour"] = stays_beijing["arrival_time_local"].dt.hour
stays_beijing["arrival_local_weekday"] = (
    stays_beijing["arrival_time_local"].dt.weekday
)

print("Timezone:", BEIJING_TZ)
print("Users in candidate cohort:", stays_beijing["user_id"].nunique())
print("Stays in candidate cohort:", len(stays_beijing))

display(
    stays_beijing[
        [
            "user_id",
            "arrival_time_utc",
            "arrival_time_local",
            "departure_time_local",
            "duration_s",
        ]
    ].head(10)
)

display(
    stays_beijing["arrival_local_hour"]
    .value_counts()
    .sort_index()
    .rename("stays")
    .to_frame()
)

print("TIMEZONE POLICY STATUS: candidate Beijing cohort ready for review")
print("Home/Office scoring remains gated until geography sensitivity is reviewed.")

### 4.3 Complete-link recurring-location audit

DBSCAN prototype cho thấy 200 m neighbor radius không bảo đảm cluster compactness. Ta chuyển sang **complete-link agglomerative clustering**.

Với complete linkage, merge distance là maximum pairwise distance giữa hai groups. Vì vậy threshold có semantics trực tiếp:

```text
threshold = 200 m
→ final cluster diameter <= 200 m
```

### Sensitivity

| threshold | locations | recurring >=2 | users with recurrence | max diameter |
|---:|---:|---:|---:|---:|
| 100 m | 1,320 | 499 | 67 | 99.95 m |
| **200 m** | **1,111** | **486** | **73** | **199.23 m** |
| 300 m | 1,007 | 473 | 73 | 297.42 m |

Từ 200→300 m không tăng thêm user coverage, chỉ merge locations thêm. 200 m vì vậy là middle engineering choice có direct compactness contract.

**Frozen CP2 v1:** per-user complete linkage, maximum cluster diameter 200 m.

In [ ]:
COMPLETE_LINK_THRESHOLDS_M = [100.0, 200.0, 300.0]
CANDIDATE_COMPLETE_LINK_M = 200.0

def pairwise_haversine_matrix_m(group):
    lat = group["latitude"].to_numpy(dtype=float)
    lon = group["longitude"].to_numpy(dtype=float)
    return np.asarray(
        haversine_m(
            lat[:, None],
            lon[:, None],
            lat[None, :],
            lon[None, :],
        ),
        dtype=float,
    )

def complete_link_user(group, threshold_m):
    g = group.sort_values("arrival_time_local", kind="stable").copy()
    n = len(g)
    if n == 1:
        g["location_id"] = 0
        return g, np.zeros((1, 1), dtype=float)

    distances = pairwise_haversine_matrix_m(g)
    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="complete",
        distance_threshold=threshold_m,
    ).fit_predict(distances)
    g["location_id"] = labels.astype(int)
    return g, distances

def summarize_complete_link(threshold_m):
    clustered_parts = []
    location_rows = []

    for user_id, group in stays_beijing.groupby("user_id", sort=True):
        clustered_user, distances = complete_link_user(group, threshold_m)
        clustered_parts.append(clustered_user)

        labels = clustered_user["location_id"].to_numpy(dtype=int)
        for location_id in np.unique(labels):
            member_idx = np.flatnonzero(labels == location_id)
            members = clustered_user.iloc[member_idx]
            diameter_m = (
                float(distances[np.ix_(member_idx, member_idx)].max())
                if len(member_idx) > 1
                else 0.0
            )
            location_rows.append({
                "user_id": user_id,
                "location_id": int(location_id),
                "latitude": float(members["latitude"].median()),
                "longitude": float(members["longitude"].median()),
                "stay_count": len(members),
                "active_local_dates": members["arrival_local_date"].nunique(),
                "total_dwell_h": members["duration_s"].sum() / 3600.0,
                "diameter_m": diameter_m,
            })

    clustered_all = pd.concat(clustered_parts, ignore_index=True)
    locations_all = pd.DataFrame(location_rows)

    recurring = locations_all["stay_count"] >= 2
    return clustered_all, locations_all, {
        "threshold_m": threshold_m,
        "locations": len(locations_all),
        "recurring_locations": int(recurring.sum()),
        "users_with_recurring_location": int(
            locations_all.loc[recurring, "user_id"].nunique()
        ),
        "median_locations_per_user": float(
            locations_all.groupby("user_id").size().median()
        ),
        "p95_diameter_m": float(locations_all["diameter_m"].quantile(0.95)),
        "max_diameter_m": float(locations_all["diameter_m"].max()),
    }

cluster_sensitivity_rows = []
cluster_artifacts = {}

for threshold_m in COMPLETE_LINK_THRESHOLDS_M:
    clustered_threshold, locations_threshold, summary = summarize_complete_link(
        threshold_m
    )
    cluster_sensitivity_rows.append(summary)
    cluster_artifacts[threshold_m] = (
        clustered_threshold,
        locations_threshold,
    )

complete_link_sensitivity = pd.DataFrame(cluster_sensitivity_rows)
display(complete_link_sensitivity)

semantic_stays, semantic_locations = cluster_artifacts[CANDIDATE_COMPLETE_LINK_M]

assert semantic_locations["diameter_m"].max() <= CANDIDATE_COMPLETE_LINK_M + 1e-6

print("Candidate complete-link threshold:", CANDIDATE_COMPLETE_LINK_M, "m")
print("Semantic locations:", len(semantic_locations))
print(
    "Recurring semantic locations (>=2 stays):",
    int((semantic_locations["stay_count"] >= 2).sum()),
)
print(
    "Users with recurring semantic location:",
    semantic_locations.loc[
        semantic_locations["stay_count"] >= 2, "user_id"
    ].nunique(),
)
print(
    "Max verified cluster diameter (m):",
    semantic_locations["diameter_m"].max(),
)

display(
    semantic_locations.sort_values(
        ["stay_count", "total_dwell_h"],
        ascending=False,
    ).head(30)
)

## 5. Frozen CP2 v1 semantic configuration

Đến đây geography và recurring-location semantics đã freeze. Home/Office vẫn cần behavioral evidence và abstention gates.

Final v1 config sau sensitivity:

```text
HOME window       21:00–06:00 local
HOME min dates    3
HOME min share    0.50
HOME min margin   0.20

OFFICE window     Mon–Fri 09:00–17:00 local
OFFICE min dates  3
OFFICE min share  0.30
OFFICE min margin 0.10

support saturation = 5 relevant dates
```

Home và Office dùng **khác gate** vì measured evidence distributions khác nhau; không giả định hai semantic scores cùng calibration.

In [ ]:
CP2_V1_CONFIG = {
    "timezone": "Asia/Shanghai",
    "beijing_reference_lat": BEIJING_CENTER[0],
    "beijing_reference_lon": BEIJING_CENTER[1],
    "beijing_radius_km": CANDIDATE_RADIUS_KM,
    "beijing_min_stay_share": CANDIDATE_MIN_STAY_SHARE,
    "beijing_min_dwell_share": CANDIDATE_MIN_DWELL_SHARE,
    "home_night_start_hour": 21,
    "home_night_end_hour": 6,
    "office_start_hour": 9,
    "office_end_hour": 17,
    "office_weekdays": [0, 1, 2, 3, 4],
    "candidate_location_complete_link_m": CANDIDATE_COMPLETE_LINK_M,
    "home_min_dates": 3,
    "home_min_share": 0.50,
    "home_min_margin": 0.20,
    "office_min_dates": 3,
    "office_min_share": 0.30,
    "office_min_margin": 0.10,
    "support_saturation_dates": 5,
}

display(pd.Series(CP2_V1_CONFIG, name="frozen_value"))
print("STATUS: CP2 v1 engineering baseline frozen; production parity check remains.")

## 6. Home / Office scoring audit

### Vì sao dùng interval overlap thay vì arrival hour?

Ví dụ stay:

```text
20:50 ───────── 21:30
         21:00 night boundary
```

Chỉ **21:00–21:30** là night evidence. Nếu nhìn arrival hour 20:50 rồi gán cả stay là “không night”, ta mất 30 phút evidence; nếu nhìn departure hour rồi gán cả stay là night, ta thêm 10 phút giả.

Vì vậy behavioral-time scoring là **interval-overlap problem**.

### Evidence ở mỗi recurring location

- relevant dwell duration;
- relevant dwell share trên toàn user;
- distinct relevant dates có >=10 phút overlap;
- top-1 vs top-2 share margin.

Denominator share bao gồm toàn bộ semantic locations của user, không chỉ recurring candidates. Nhờ đó scattered non-recurring dwell không biến mất khỏi denominator.

### Same-location semantics

HOME và OFFICE **không bị ép phải khác location**. Nếu một location dẫn đầu cả night và weekday-daytime evidence, notebook report ambiguity đó thay vì invent location thứ hai.

In [ ]:
HOME_NIGHT_START_HOUR = 21
HOME_NIGHT_END_HOUR = 6
OFFICE_START_HOUR = 9
OFFICE_END_HOUR = 17
OFFICE_WEEKDAYS = {0, 1, 2, 3, 4}

MIN_RELEVANT_DATE_OVERLAP_S = 10 * 60
MIN_RELEVANT_DATES = 2

def interval_overlap_s(start, end, window_start, window_end):
    overlap_start = max(start, window_start)
    overlap_end = min(end, window_end)
    if overlap_end <= overlap_start:
        return 0.0
    return float((overlap_end - overlap_start).total_seconds())

def stay_window_contributions(row):
    start = row.arrival_time_local
    end = row.departure_time_local

    night_rows = []
    office_rows = []

    # A stay shortly after midnight can overlap the night window that started
    # on the previous local date, so include one padded date before arrival.
    day = start.normalize() - pd.Timedelta(days=1)
    last_day = end.normalize()

    while day <= last_day:
        night_start = day + pd.Timedelta(hours=HOME_NIGHT_START_HOUR)
        night_end = day + pd.Timedelta(days=1, hours=HOME_NIGHT_END_HOUR)
        night_s = interval_overlap_s(start, end, night_start, night_end)
        if night_s > 0:
            night_rows.append(
                {
                    "user_id": row.user_id,
                    "location_id": int(row.location_id),
                    "behavior_date": night_start.date(),
                    "overlap_s": night_s,
                }
            )

        if day.weekday() in OFFICE_WEEKDAYS:
            office_start = day + pd.Timedelta(hours=OFFICE_START_HOUR)
            office_end = day + pd.Timedelta(hours=OFFICE_END_HOUR)
            office_s = interval_overlap_s(start, end, office_start, office_end)
            if office_s > 0:
                office_rows.append(
                    {
                        "user_id": row.user_id,
                        "location_id": int(row.location_id),
                        "behavior_date": office_start.date(),
                        "overlap_s": office_s,
                    }
                )

        day += pd.Timedelta(days=1)

    return night_rows, office_rows

night_rows = []
office_rows = []

for row in semantic_stays.itertuples(index=False):
    nr, orows = stay_window_contributions(row)
    night_rows.extend(nr)
    office_rows.extend(orows)

night_contrib = pd.DataFrame(
    night_rows,
    columns=["user_id", "location_id", "behavior_date", "overlap_s"],
)
office_contrib = pd.DataFrame(
    office_rows,
    columns=["user_id", "location_id", "behavior_date", "overlap_s"],
)

def aggregate_relevant_window(contrib, prefix):
    if contrib.empty:
        return pd.DataFrame(
            columns=[
                "user_id",
                "location_id",
                f"{prefix}_dwell_s",
                f"{prefix}_dates",
            ]
        )

    per_date = (
        contrib.groupby(["user_id", "location_id", "behavior_date"], as_index=False)
        ["overlap_s"]
        .sum()
    )

    dwell = (
        per_date.groupby(["user_id", "location_id"], as_index=False)["overlap_s"]
        .sum()
        .rename(columns={"overlap_s": f"{prefix}_dwell_s"})
    )

    supported_dates = (
        per_date.loc[per_date["overlap_s"] >= MIN_RELEVANT_DATE_OVERLAP_S]
        .groupby(["user_id", "location_id"], as_index=False)["behavior_date"]
        .nunique()
        .rename(columns={"behavior_date": f"{prefix}_dates"})
    )

    return dwell.merge(
        supported_dates,
        on=["user_id", "location_id"],
        how="left",
    ).fillna({f"{prefix}_dates": 0})

night_features = aggregate_relevant_window(night_contrib, "night")
office_features = aggregate_relevant_window(office_contrib, "office")

semantic_features = (
    semantic_locations.merge(
        night_features,
        on=["user_id", "location_id"],
        how="left",
    )
    .merge(
        office_features,
        on=["user_id", "location_id"],
        how="left",
    )
)

for col in ["night_dwell_s", "night_dates", "office_dwell_s", "office_dates"]:
    semantic_features[col] = semantic_features[col].fillna(0)

semantic_features["night_dates"] = semantic_features["night_dates"].astype(int)
semantic_features["office_dates"] = semantic_features["office_dates"].astype(int)

user_night_total = semantic_features.groupby("user_id")["night_dwell_s"].transform("sum")
user_office_total = semantic_features.groupby("user_id")["office_dwell_s"].transform("sum")

semantic_features["night_dwell_share"] = np.where(
    user_night_total > 0,
    semantic_features["night_dwell_s"] / user_night_total,
    0.0,
)
semantic_features["office_dwell_share"] = np.where(
    user_office_total > 0,
    semantic_features["office_dwell_s"] / user_office_total,
    0.0,
)

semantic_features["night_dwell_h"] = semantic_features["night_dwell_s"] / 3600.0
semantic_features["office_dwell_h"] = semantic_features["office_dwell_s"] / 3600.0

def rank_semantic_candidates(
    features,
    *,
    share_col,
    dates_col,
    dwell_col,
    label,
):
    eligible = features[
        (features["stay_count"] >= 2)
        & (features[dates_col] >= MIN_RELEVANT_DATES)
        & (features[dwell_col] > 0)
    ].copy()

    if eligible.empty:
        return eligible

    eligible = eligible.sort_values(
        ["user_id", share_col, dates_col, dwell_col, "stay_count"],
        ascending=[True, False, False, False, False],
        kind="stable",
    )
    eligible[f"{label}_rank"] = eligible.groupby("user_id").cumcount() + 1
    return eligible

home_ranked = rank_semantic_candidates(
    semantic_features,
    share_col="night_dwell_share",
    dates_col="night_dates",
    dwell_col="night_dwell_s",
    label="home",
)
office_ranked = rank_semantic_candidates(
    semantic_features,
    share_col="office_dwell_share",
    dates_col="office_dates",
    dwell_col="office_dwell_s",
    label="office",
)

def top_with_margin(ranked, *, label, share_col):
    if ranked.empty:
        return pd.DataFrame()

    top1 = ranked[ranked[f"{label}_rank"] == 1].copy()
    second = (
        ranked[ranked[f"{label}_rank"] == 2][["user_id", share_col]]
        .rename(columns={share_col: f"{label}_second_share"})
    )
    top1 = top1.merge(second, on="user_id", how="left")
    top1[f"{label}_second_share"] = top1[f"{label}_second_share"].fillna(0.0)
    top1[f"{label}_share_margin"] = (
        top1[share_col] - top1[f"{label}_second_share"]
    )
    return top1

home_top = top_with_margin(
    home_ranked,
    label="home",
    share_col="night_dwell_share",
)
office_top = top_with_margin(
    office_ranked,
    label="office",
    share_col="office_dwell_share",
)

candidate_users = set(stays_beijing["user_id"].unique())
home_users = set(home_top["user_id"]) if not home_top.empty else set()
office_users = set(office_top["user_id"]) if not office_top.empty else set()
both_users = home_users & office_users

print("Beijing semantic cohort users:", len(candidate_users))
print("Users with recurring semantic location:", semantic_locations.loc[
    semantic_locations["stay_count"] >= 2, "user_id"
].nunique())
print("Users with supported HOME candidate:", len(home_users))
print("Users with supported OFFICE candidate:", len(office_users))
print("Users with both candidates:", len(both_users))
print("Users abstaining from HOME:", len(candidate_users - home_users))
print("Users abstaining from OFFICE:", len(candidate_users - office_users))

if both_users:
    paired = (
        home_top[home_top["user_id"].isin(both_users)][
            ["user_id", "location_id"]
        ]
        .rename(columns={"location_id": "home_location_id"})
        .merge(
            office_top[office_top["user_id"].isin(both_users)][
                ["user_id", "location_id"]
            ].rename(columns={"location_id": "office_location_id"}),
            on="user_id",
        )
    )
    paired["same_location_candidate"] = (
        paired["home_location_id"] == paired["office_location_id"]
    )
    print(
        "Both-candidate users with same leading location:",
        int(paired["same_location_candidate"].sum()),
        "/",
        len(paired),
    )

def candidate_distribution(top, cols):
    if top.empty:
        return pd.DataFrame()
    return top[cols].describe(
        percentiles=[.1, .25, .5, .75, .9, .95]
    )

print("\nHOME top-candidate evidence:")
display(
    candidate_distribution(
        home_top,
        [
            "night_dwell_share",
            "home_share_margin",
            "night_dates",
            "night_dwell_h",
            "stay_count",
        ],
    )
)

print("\nOFFICE top-candidate evidence:")
display(
    candidate_distribution(
        office_top,
        [
            "office_dwell_share",
            "office_share_margin",
            "office_dates",
            "office_dwell_h",
            "stay_count",
        ],
    )
)

print("\nSample HOME candidates (no coordinates displayed):")
display(
    home_top[
        [
            "user_id",
            "location_id",
            "stay_count",
            "active_local_dates",
            "night_dates",
            "night_dwell_h",
            "night_dwell_share",
            "home_share_margin",
        ]
    ].head(20)
)

print("\nSample OFFICE candidates (no coordinates displayed):")
display(
    office_top[
        [
            "user_id",
            "location_id",
            "stay_count",
            "active_local_dates",
            "office_dates",
            "office_dwell_h",
            "office_dwell_share",
            "office_share_margin",
        ]
    ].head(20)
)

### 6.0 First measured scoring result

Trước khi áp final emission gates, chỉ yêu cầu recurrence + >=2 relevant dates:

| metric | users |
|---|---:|
| semantic cohort | 97 |
| recurring-location users | 73 |
| supported HOME candidate | 47 |
| supported OFFICE candidate | 40 |
| both | 27 |
| same leading location trong both | 7 / 27 |

Home evidence mạnh hơn Office:

| evidence | HOME median | OFFICE median |
|---|---:|---:|
| relevant dwell share | 0.635 | 0.357 |
| top-two margin | 0.513 | 0.243 |
| relevant dates | 4 | 3 |
| relevant dwell | 7.04 h | 2.94 h |

Low tail có candidate share ~0.09 và gần-zero margin, nên **ranking alone không đủ**. Cần explicit abstention gate.

### 6.1 Bounded scoring sensitivity

Không có ground truth để maximize accuracy, nên ta kiểm tra hai loại robustness.

#### A. Time-window stability

Home: `20–06 / 21–06 / 22–06`  
Office: `08–17 / 09–17 / 09–18`

Metric chính là **same top location rate** trên shared users.

Measured:

- Home 20–06 vs 21–06: **93.6%** same top;
- Home 22–06 vs 21–06: **90.5%**;
- Office 08–17 vs 09–17: **95.0%**;
- Office 09–18 vs 09–17: **92.5%**.

Baseline windows vì vậy không nằm ở unstable boundary.

#### B. Emission / abstention grid

Home grid review: dates `2/3/5`, share `0.4/0.5/0.6`, margin `0.1/0.2/0.3`.  
Office grid review: dates `2/3/5`, share `0.2/0.3/0.4`, margin `0.05/0.10/0.20`.

Chọn **middle sensitivity setting**:

```text
HOME   3 dates / 0.50 share / 0.20 margin → 27 users
OFFICE 3 dates / 0.30 share / 0.10 margin → 16 users
```

Đây là engineering abstention gates, không phải supervised-optimal cutoffs.

In [ ]:
def build_semantic_features_for_windows(
    *,
    home_start_hour,
    home_end_hour,
    office_start_hour,
    office_end_hour,
):
    night_rows = []
    office_rows = []

    for row in semantic_stays.itertuples(index=False):
        start = row.arrival_time_local
        end = row.departure_time_local

        day = start.normalize() - pd.Timedelta(days=1)
        last_day = end.normalize()

        while day <= last_day:
            night_start = day + pd.Timedelta(hours=home_start_hour)
            night_end = day + pd.Timedelta(days=1, hours=home_end_hour)
            night_s = interval_overlap_s(start, end, night_start, night_end)
            if night_s > 0:
                night_rows.append(
                    {
                        "user_id": row.user_id,
                        "location_id": int(row.location_id),
                        "behavior_date": night_start.date(),
                        "overlap_s": night_s,
                    }
                )

            if day.weekday() in OFFICE_WEEKDAYS:
                office_start = day + pd.Timedelta(hours=office_start_hour)
                office_end = day + pd.Timedelta(hours=office_end_hour)
                office_s = interval_overlap_s(start, end, office_start, office_end)
                if office_s > 0:
                    office_rows.append(
                        {
                            "user_id": row.user_id,
                            "location_id": int(row.location_id),
                            "behavior_date": office_start.date(),
                            "overlap_s": office_s,
                        }
                    )

            day += pd.Timedelta(days=1)

    night_contrib_local = pd.DataFrame(
        night_rows,
        columns=["user_id", "location_id", "behavior_date", "overlap_s"],
    )
    office_contrib_local = pd.DataFrame(
        office_rows,
        columns=["user_id", "location_id", "behavior_date", "overlap_s"],
    )

    night_features_local = aggregate_relevant_window(
        night_contrib_local,
        "night",
    )
    office_features_local = aggregate_relevant_window(
        office_contrib_local,
        "office",
    )

    features = (
        semantic_locations.merge(
            night_features_local,
            on=["user_id", "location_id"],
            how="left",
        )
        .merge(
            office_features_local,
            on=["user_id", "location_id"],
            how="left",
        )
    )

    for col in ["night_dwell_s", "night_dates", "office_dwell_s", "office_dates"]:
        features[col] = features[col].fillna(0)

    features["night_dates"] = features["night_dates"].astype(int)
    features["office_dates"] = features["office_dates"].astype(int)

    user_night_total = features.groupby("user_id")["night_dwell_s"].transform("sum")
    user_office_total = features.groupby("user_id")["office_dwell_s"].transform("sum")

    features["night_dwell_share"] = np.where(
        user_night_total > 0,
        features["night_dwell_s"] / user_night_total,
        0.0,
    )
    features["office_dwell_share"] = np.where(
        user_office_total > 0,
        features["office_dwell_s"] / user_office_total,
        0.0,
    )

    features["night_dwell_h"] = features["night_dwell_s"] / 3600.0
    features["office_dwell_h"] = features["office_dwell_s"] / 3600.0
    return features


def top_candidates_for(
    features,
    *,
    label,
    min_dates,
):
    if label == "home":
        share_col = "night_dwell_share"
        dates_col = "night_dates"
        dwell_col = "night_dwell_s"
    elif label == "office":
        share_col = "office_dwell_share"
        dates_col = "office_dates"
        dwell_col = "office_dwell_s"
    else:
        raise ValueError(label)

    ranked = features[
        (features["stay_count"] >= 2)
        & (features[dates_col] >= min_dates)
        & (features[dwell_col] > 0)
    ].copy()

    if ranked.empty:
        return ranked

    ranked = ranked.sort_values(
        ["user_id", share_col, dates_col, dwell_col, "stay_count"],
        ascending=[True, False, False, False, False],
        kind="stable",
    )
    ranked[f"{label}_rank"] = ranked.groupby("user_id").cumcount() + 1
    return top_with_margin(
        ranked,
        label=label,
        share_col=share_col,
    )


def window_stability_row(
    top,
    *,
    baseline_top,
    label,
    variant,
    share_col,
    margin_col,
    dates_col,
):
    if top.empty:
        return {
            "variant": variant,
            "supported_users": 0,
            "shared_with_baseline": 0,
            "same_top_location_rate": np.nan,
            "median_share": np.nan,
            "median_margin": np.nan,
            "median_dates": np.nan,
        }

    current = top[["user_id", "location_id"]].rename(
        columns={"location_id": "current_location_id"}
    )
    base = baseline_top[["user_id", "location_id"]].rename(
        columns={"location_id": "baseline_location_id"}
    )
    shared = current.merge(base, on="user_id", how="inner")

    same_rate = (
        float(
            (shared["current_location_id"] == shared["baseline_location_id"]).mean()
        )
        if len(shared)
        else np.nan
    )

    return {
        "variant": variant,
        "supported_users": len(top),
        "shared_with_baseline": len(shared),
        "same_top_location_rate": same_rate,
        "median_share": float(top[share_col].median()),
        "median_margin": float(top[margin_col].median()),
        "median_dates": float(top[dates_col].median()),
    }


HOME_WINDOW_VARIANTS = [
    ("20-06", 20, 6),
    ("21-06", 21, 6),
    ("22-06", 22, 6),
]
OFFICE_WINDOW_VARIANTS = [
    ("08-17", 8, 17),
    ("09-17", 9, 17),
    ("09-18", 9, 18),
]

home_window_rows = []
for name, start_hour, end_hour in HOME_WINDOW_VARIANTS:
    features_variant = build_semantic_features_for_windows(
        home_start_hour=start_hour,
        home_end_hour=end_hour,
        office_start_hour=OFFICE_START_HOUR,
        office_end_hour=OFFICE_END_HOUR,
    )
    top_variant = top_candidates_for(
        features_variant,
        label="home",
        min_dates=2,
    )
    home_window_rows.append(
        window_stability_row(
            top_variant,
            baseline_top=home_top,
            label="home",
            variant=name,
            share_col="night_dwell_share",
            margin_col="home_share_margin",
            dates_col="night_dates",
        )
    )

office_window_rows = []
for name, start_hour, end_hour in OFFICE_WINDOW_VARIANTS:
    features_variant = build_semantic_features_for_windows(
        home_start_hour=HOME_NIGHT_START_HOUR,
        home_end_hour=HOME_NIGHT_END_HOUR,
        office_start_hour=start_hour,
        office_end_hour=end_hour,
    )
    top_variant = top_candidates_for(
        features_variant,
        label="office",
        min_dates=2,
    )
    office_window_rows.append(
        window_stability_row(
            top_variant,
            baseline_top=office_top,
            label="office",
            variant=name,
            share_col="office_dwell_share",
            margin_col="office_share_margin",
            dates_col="office_dates",
        )
    )

print("HOME window stability:")
display(pd.DataFrame(home_window_rows))

print("OFFICE window stability:")
display(pd.DataFrame(office_window_rows))


def emission_grid(
    features,
    *,
    label,
    min_dates_values,
    min_share_values,
    min_margin_values,
):
    if label == "home":
        share_col = "night_dwell_share"
        margin_col = "home_share_margin"
    elif label == "office":
        share_col = "office_dwell_share"
        margin_col = "office_share_margin"
    else:
        raise ValueError(label)

    rows = []
    for min_dates in min_dates_values:
        top = top_candidates_for(
            features,
            label=label,
            min_dates=min_dates,
        )

        for min_share in min_share_values:
            for min_margin in min_margin_values:
                emitted = top[
                    (top[share_col] >= min_share)
                    & (top[margin_col] >= min_margin)
                ]
                rows.append(
                    {
                        "min_dates": min_dates,
                        "min_share": min_share,
                        "min_margin": min_margin,
                        "emitted_users": len(emitted),
                        "cohort_coverage": len(emitted) / len(candidate_users),
                        "recurring_user_coverage": len(emitted) / 73.0,
                    }
                )
    return pd.DataFrame(rows)


home_emission_sensitivity = emission_grid(
    semantic_features,
    label="home",
    min_dates_values=[2, 3, 5],
    min_share_values=[0.4, 0.5, 0.6],
    min_margin_values=[0.1, 0.2, 0.3],
)

office_emission_sensitivity = emission_grid(
    semantic_features,
    label="office",
    min_dates_values=[2, 3, 5],
    min_share_values=[0.2, 0.3, 0.4],
    min_margin_values=[0.05, 0.10, 0.20],
)

print("HOME emission sensitivity:")
display(home_emission_sensitivity)

print("OFFICE emission sensitivity:")
display(office_emission_sensitivity)

### Cách đọc share, margin và support

`relevant_dwell_share` trả lời: location này chiếm bao nhiêu phần evidence window của user.

`share_margin` trả lời: top location có tách khỏi runner-up rõ không.

`relevant_dates` trả lời: evidence có repeat qua nhiều ngày hay chỉ đến từ một episode dài.

Ba signals bổ sung cho nhau:

```text
share cao nhưng margin thấp → hai locations cạnh tranh gần ngang nhau
margin cao nhưng 1–2 dates → evidence còn mỏng
nhiều dates nhưng share thấp → behavior phân tán
```

Vì vậy final emission dùng **gate**, còn aggregate confidence chỉ là evidence-strength summary.

### 6.2 Frozen CP2 v1 scoring decision

Bounded window sensitivity kept the baseline top location stable for >90% of shared users under every neighboring window tested.

Frozen engineering baseline:

```text
HOME
window:     21:00–06:00
min dates:  3
min share:  0.50
min margin: 0.20
measured emitted users: 27

OFFICE
window:     weekdays 09:00–17:00
min dates:  3
min share:  0.30
min margin: 0.10
measured emitted users: 16
```

Heuristic evidence strength:

```text
support_factor = min(relevant_dates / 5, 1)
evidence_strength = (share + margin + support_factor) / 3
```

This score is **not a calibrated probability**. Raw share, margin, relevant dates and dwell remain part of the output contract.

Measured emission counts `27 HOME / 16 OFFICE` là **coverage dưới frozen heuristic**, không phải số HOME/OFFICE đúng theo ground truth.

### 6.3 Production parity smoke check

Notebook exploration và production code phải đồng ý trên cùng full cached stay table.

Cell bên dưới gọi trực tiếp:

```python
infer_home_office(stays, config=HomeOfficeConfig())
```

nên đây là **assembled production path**, không phải copy notebook heuristic.

Expected frozen result:

```text
HOME    27
OFFICE  16
```

Measured final parity:

- **27 HOME**;
- **16 OFFICE**;
- **43 emitted rows**;
- **36 unique users**;
- parity assertion: **PASS**.

43 rows trên 36 users nghĩa là một số users emit cả HOME và OFFICE; không phải 43 distinct users.

In [ ]:
production_config = HomeOfficeConfig()
production_labels = infer_home_office(stays, config=production_config)

production_counts = (
    production_labels["label"]
    .value_counts()
    .reindex(["HOME", "OFFICE"], fill_value=0)
)

display(production_counts.rename("emitted_users").to_frame())

assert int(production_counts["HOME"]) == 27
assert int(production_counts["OFFICE"]) == 16
assert production_labels["evidence_strength"].between(0.0, 1.0).all()

print("Production parity smoke check: OK")
print("Rows:", len(production_labels))
print("Unique users:", production_labels["user_id"].nunique())

## 7. Kết luận CP2 v1

### Frozen pipeline

```text
5,821 CP1 stays
   ↓ history sufficiency
136 users with stays
   ↓ Beijing 100km + 80/80 cohort
97 eligible users / 4,197 in-region stays
   ↓ complete-link max diameter 200m
1,111 semantic locations / 73 users with recurrence
   ↓ interval-overlap behavioral evidence
HOME 21–06 / OFFICE weekday 09–17
   ↓ separate abstention gates
27 HOME / 16 OFFICE
   ↓ production parity
43 rows / 36 unique users
```

### Evidence strength

```text
support_factor = min(relevant_dates / 5, 1)
evidence_strength = (share + margin + support_factor) / 3
```

Score này bounded `[0,1]` nhưng **không phải probability**. Raw share, margin, dates và dwell vẫn phải được giữ để giải thích prediction.

### Những giới hạn phải nhớ

- v1 chỉ cover Beijing-focused cohort, không phải global timezone solution;
- GeoLife không có direct HOME/OFFICE ground truth;
- thresholds là engineering baselines dựa trên bounded sensitivity, không phải empirical optimum;
- precise inferred HOME/OFFICE coordinates là sensitive derived data và không được commit;
- user ngoài cohort hoặc evidence yếu phải được phép abstain.

### Validation status

- CP1 stay parity: PASS;
- geography sensitivity: reviewed;
- recurring-location sensitivity: reviewed;
- behavioral-window sensitivity: reviewed;
- RED acceptance tests: PASS;
- production API CI: PASS;
- full-release production parity `27/16`: PASS.

CP2 v1 vì vậy đủ điều kiện merge như một **transparent heuristic baseline** cho các checkpoint/modeling bước sau.